In [ ]:
from pyspark.sql import SparkSession
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import ast
from sklearn.ensemble import RandomForestRegressor
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re
import geopandas as gpd
import folium


spark = (
    SparkSession.builder.appName("MAST30034 Project 2")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

In [ ]:
rental_data = pd.read_csv('rea_data/domain/Data/vic_rentals_all_cleaned.csv')
income_data = pd.read_csv('data/external/income_sa2.csv')
postcodes = pd.read_csv('data/external/australian_postcodes.csv')
postcodes_gdf = gpd.read_file("Order_J2W0RK/gda2020_vicgrid/esrishape/whole_of_dataset/victoria/VMADMIN/POSTCODE_POLYGON.shp")
postcodes = postcodes[['SA2_NAME_2021', 'postcode']]

In [ ]:
df_postcodes_unique = postcodes.drop_duplicates(subset="SA2_NAME_2021")
income_postcode = income_data.merge(df_postcodes_unique, left_on='sa2_name', right_on='SA2_NAME_2021', how='inner')
missing_suburbs = income_data[~income_data["sa2_name"].isin(postcodes["SA2_NAME_2021"])]
print(missing_suburbs.shape)
print(income_data.shape)
print(income_postcode.shape)

In [ ]:
rent_by_postcode = (
    rental_data
    .groupby("postcode", as_index=False)
    .agg(
        mean_weekly_rent=("weekly_rent", "mean"),
        rental_count=("weekly_rent", "count")
    )
    .sort_values(by='mean_weekly_rent', ascending=False)
)

print(rent_by_postcode.head(10))

df_final = income_postcode.merge(rent_by_postcode, on="postcode", how="inner")
print(df_final[['postcode', 'mean_weekly_rent', 'median_hh_income']].head(10))

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(df_final["median_hh_income"], df_final["mean_weekly_rent"], alpha=0.7)

plt.title("Median Income vs Mean Weekly Rent by Postcode", fontsize=14)
plt.xlabel("Median Income ($)", fontsize=12)
plt.ylabel("Mean Weekly Rent ($)", fontsize=12)
plt.grid(True, linestyle="-", alpha=0.4)

plt.show()

In [ ]:
rent_by_postcode = (
    rental_data
    .groupby("postcode", as_index=False)
    .agg(
        suburb=("suburb", "first"),
        mean_bedrooms=("bedrooms", "mean"),
        mean_rent=('weekly_rent', 'mean'),
        rental_count=("weekly_rent", "count")
    )
)
rent_by_postcode['rent_bed_ratio'] = rent_by_postcode['mean_rent']/rent_by_postcode['mean_bedrooms']

df_final['rai'] = 100*df_final['median_hh_income']/(df_final['mean_weekly_rent']/0.3)
df_final = df_final[['postcode', 'SA2_NAME_2021', 'mean_weekly_rent', 'median_hh_income', 'rental_count', 'rai']]
print(df_final[df_final['rental_count'] > 10].sort_values(by='rai', ascending=True).head(10))
print(df_final[df_final['rental_count'] > 10].sort_values(by='rai', ascending=False).head(10))

print(rent_by_postcode[rent_by_postcode['rental_count'] > 10].sort_values(by='rent_bed_ratio', ascending=True).head(10))
print(rent_by_postcode[rent_by_postcode['rental_count'] > 10].sort_values(by='rent_bed_ratio', ascending=False).head(10))

In [ ]:
postcodes_gdf['POSTCODE'] = postcodes_gdf['POSTCODE'].astype(int)
postcodes_gdf = postcodes_gdf[['POSTCODE', 'geometry']]
geo_rent = postcodes_gdf.merge(rent_by_postcode, left_on="POSTCODE", right_on="postcode", how="left")

map_center = [-37.8136, 144.9631]

m = folium.Map(location=map_center, zoom_start=11)

# Choropleth map
folium.Choropleth(
    geo_data=geo_rent,
    name="choropleth",
    data=geo_rent,
    columns=["POSTCODE", "rent_bed_ratio"],
    key_on="feature.properties.POSTCODE",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Rent Bed Ratio",
).add_to(m)

m

In [ ]:
postcodes_gdf = gpd.read_file("Order_J2W0RK/gda2020_vicgrid/esrishape/whole_of_dataset/victoria/VMADMIN/POSTCODE_POLYGON.shp")
postcodes_gdf['POSTCODE'] = postcodes_gdf['POSTCODE'].astype(int)
postcodes_gdf = postcodes_gdf[['POSTCODE', 'geometry']]
geo_rent_rai = postcodes_gdf.merge(df_final, left_on="POSTCODE", right_on="postcode", how="left")

map_center = [-37.8136, 144.9631]

m_2 = folium.Map(location=map_center, zoom_start=11)

# Choropleth map
folium.Choropleth(
    geo_data=geo_rent_rai,
    name="choropleth",
    data=geo_rent_rai,
    columns=["POSTCODE", "rai"],
    key_on="feature.properties.POSTCODE",
    fill_color="YlOrRd_r",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="RAI",
).add_to(m_2)

m_2